<a href="https://colab.research.google.com/github/LIBY70/Data-Analysis/blob/main/ida_week4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab4: pandas를 이용한 누락 데이터 처리와 데이터 집계

본 실습 자료는 『배워서 바로 써먹는 데이터 분석with 파이썬』 (설진욱, 생능북스)를 참고하여 제작되었습니다.

## 1. 누락 데이터 처리

### 누락 데이터 확인

| 항목 | 설명 |
|------|------|
| `isnull()` | 해당 요소의 값이 NaN인 경우 True, 값이 존재하는 경우 False 반환 / 파이썬의 None도 NaN으로 취급 |
| `notnull()` | `isnull()`과 True/False를 반대로 반환 / `myseries[myseries.notnull()]`로 참인 항목만 출력 가능 |

In [ ]:
import numpy as np
from pandas import Series, DataFrame

myseries = Series([1, 2, np.nan, 4, 5])
myseries

- `isnull()`: 값이 NaN이면 True 반환 (파이썬의 None도 NaN으로 취급)
- `notnull()`: `isnull()`의 반대
- `myseries[myseries.notnull()]`: 시리즈에서 NaN을 제외한 데이터만 추출

In [ ]:
myseries.isnull()

In [ ]:
myseries.notnull()

In [ ]:
# NaN을 제외한 데이터만 추출
myseries[myseries.notnull()]

- `dropna()`: 누락된 데이터가 있는 행과 열을 제외시키고, 나머지 행 또는 열을 조회 (기본값: NaN이 하나라도 있는 모든 행 배제)

In [ ]:
myseries.dropna()

`excel02.csv`: 일부 데이터에 NaN이 포함된 샘플 파일

In [ ]:
import pandas as pd

myframe = pd.read_csv('excel02.csv', index_col='이름')
myframe

- `dropna()` 기본값: NaN이 하나라도 있는 행 배제 → '강감찬'(NaN 없음)만 조회

In [ ]:
myframe.dropna()

- `how='all'`: 모든 값이 NaN인 행만 배제 → '박영희'(전 과목 미응시)만 제외

In [ ]:
myframe.dropna(how='all')

- `subset`: 특정 열에 대해서만 NaN 여부 체크 → '영어' 컬럼에 NaN이 있는 행만 삭제

In [ ]:
myframe.dropna(subset=['영어'])

- `axis=1`: 열 방향으로 적용 / `how='all'`: 모든 값이 NaN인 열만 삭제 → 해당 열 없으므로 제거 없음

In [ ]:
myframe.dropna(axis=1, how='all')

- '강감찬'과 '홍길동'의 '국어' 점수를 NaN으로 변경 → '국어' 열 전체가 NaN

In [ ]:
myframe.loc[['강감찬', '홍길동'], '국어'] = np.nan
myframe

- `axis=1`, `how='all'`: '국어' 열 전체가 NaN이므로 '국어' 열 삭제

In [ ]:
myframe.dropna(axis=1, how='all')

- `thresh=2`: NaN이 아닌 데이터가 2개 이상인 행만 조회

In [ ]:
myframe.dropna(thresh=2)

- `axis=1` 기본값(`how='any'`): NaN이 하나라도 있는 열 모두 제거 → 빈 DataFrame 반환

In [ ]:
myframe.dropna(axis=1)

### 누락된 데이터 값 채우기

`fillna()`: 누락된 데이터를 다른 값으로 채워 넣는 함수

In [ ]:
import pandas as pd
import numpy as np

myframe = pd.read_csv('excel02.csv', index_col='이름')
myframe

- `inplace=False` (기본값): 원본을 변경하지 않고 결과만 반환

In [ ]:
myframe.fillna(0, inplace=False)

In [ ]:
# 원본은 변경되지 않음
myframe

- `inplace=True`: 복사본을 생성하지 않고 객체 자체를 변경

In [ ]:
myframe.fillna(0, inplace=True)
myframe

In [ ]:
myframe = pd.read_csv('excel02.csv', index_col='이름')
myframe

- 딕셔너리로 컬럼별 다른 값으로 치환 가능 → 국어: 15, 영어: 25, 수학: 35

In [ ]:
myframe.fillna({'국어': 15, '영어': 25, '수학': 35})

In [ ]:
myframe = pd.read_csv('excel02.csv', index_col='이름')
myframe

- NaN을 해당 컬럼의 평균값으로 대체: `mean()`으로 평균 계산 후 딕셔너리 형식으로 치환

In [ ]:
fill_values = {
    '국어': myframe['국어'].mean(),
    '영어': myframe['영어'].mean(),
    '수학': myframe['수학'].mean()
}
myframe.fillna(fill_values)

## 2. 데이터 집계

### 데이터 집계 관련 함수

| 항목 | 설명 |
|------|------|
| `sum()` | 배열 전체 혹은 특정 축에 대한 모든 원소의 합 계산 / 크기가 0인 배열에 대한 연산 결과는 0 |
| `mean()` | 산술 평균 / 크기가 0인 배열에 대한 연산 결과는 NaN |
| `std()`, `var()` | 표준 편차와 분산 / 선택적으로 자유도 설정 가능, 분모의 기본값은 n |
| `min()`, `max()` | 최솟값, 최댓값 |
| `argmin()`, `argmax()` | 최소 원소의 색인 값, 최대 원소의 색인 값 |
| `cumsum()` | 각 원소의 누적 합 (cumulative sum) |
| `cumprod()` | 각 원소의 누적 곱 |
| `describe()` | 시리즈에도 사용 가능 / 기술 통계 정보 반환 |

### 기술 통계 계산과 요약

In [ ]:
import numpy as np
from pandas import Series, DataFrame

myframe = DataFrame(
    {
        '국어': [75, np.nan, 90, 95],
        '영어': [np.nan, 70, 85, 80],
        '수학': [83, 88, 92, np.nan]
    },
    index=['이순신', '김유신', '강감찬', '계백']
)
myframe

- 집계 함수 기본값: NaN 배제 후 연산 / `sum()`: 각 컬럼의 합을 담은 시리즈 반환

In [ ]:
myframe.sum()

- `axis=1`: 행 방향으로 합산

In [ ]:
myframe.sum(axis=1)

- `skipna=False`: NaN이 하나라도 있으면 연산 결과를 NaN으로 반환

In [ ]:
myframe.sum(axis=1, skipna=False)

- `skipna=True`: NaN 무시하고 연산 수행

In [ ]:
myframe.sum(axis=1, skipna=True)

- `max(skipna=False)`: NaN 포함 시 결과도 NaN → '이순신'(NaN 포함)은 NaN 반환

In [ ]:
myframe.max(axis=1, skipna=False)

- `max(skipna=True)`: NaN 배제 후 최댓값

In [ ]:
myframe.max(axis=1, skipna=True)

- `idxmax()`: 최댓값을 가진 색인 반환 → 국어 최고점: '계백'

In [ ]:
myframe.idxmax()

누적 관련 함수
- `cumsum()`: 누적 합 (cumulative sums)
- `cumprod()`: 누적 곱 (cumulative products)
- `cummin()`: 누적 최솟값 (cumulative minimal)
- `cummax()`: 누적 최댓값 (cumulative maximal)

In [ ]:
myframe

In [ ]:
myframe.cumsum()

In [ ]:
myframe.cumprod()

In [ ]:
myframe.cummin()

In [ ]:
myframe.cummax()

NaN 값 처리 방법
- 기본값 정의 후 대입
- NaN이 아닌 값들의 평균값으로 대체: `mean()`으로 평균 계산 후 딕셔너리 형식으로 치환

In [ ]:
fill_values = {
    '국어': myframe['국어'].mean(),
    '영어': myframe['영어'].mean(),
    '수학': myframe['수학'].mean()
}
newframe = myframe.fillna(fill_values)
newframe

`describe()`: 한 번에 여러 통계 결과 반환 (시리즈/데이터프레임 모두 사용 가능)
- `count`: 빈도 수
- `mean`: 평균값
- `std`: 표준 편차
- `min` ~ `max`: 4분위수 (전체 데이터를 4등분하여 25, 50, 75, 100%에 해당하는 값)

In [ ]:
myframe.describe()

- NaN을 평균값으로 대체한 데이터프레임의 통계

In [ ]:
newframe.describe()